In [2]:
pip install pandas scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import os

manual_df = pd.read_csv("data/vid3_v1 (1).csv")
raw_df = pd.read_csv("data/cleaned/vid2_v1.csv")


for round_num in range(1, 6):
    print(f"\n🔁 Vòng Active Learning {round_num}")

    vectorizer = TfidfVectorizer(max_features=3000)
    X_labeled = vectorizer.fit_transform(manual_df["Comment"])
    y_labeled = manual_df["Label"].astype(str)

    X_pool = vectorizer.transform(raw_df["Comment"])

    model = LogisticRegression(max_iter=1000)
    model.fit(X_labeled, y_labeled)

    probs = model.predict_proba(X_pool)
    uncertainties = 1 - np.max(probs, axis=1)
    query_idx = uncertainties.argsort()[-50:]

    query_samples = raw_df.iloc[query_idx].copy()
    query_samples["Label"] = ""  

    query_file = f"data/active_learning/query_round_{round_num}.csv"
    query_samples.to_csv(query_file, index=False, encoding="utf-8-sig")
    print(f"📤 Đã xuất 50 comment chưa gán nhãn vào {query_file}")
    input("📝 Gán nhãn xong rồi nhấn Enter để tiếp tục...")

    labeled_query = pd.read_csv(query_file)
    labeled_query = labeled_query.dropna(subset=["Label", "Comment"])

    manual_df = pd.concat([manual_df, labeled_query], ignore_index=True)
    manual_df.to_csv("data/manual_label.csv", index=False, encoding="utf-8-sig")
    print("✅ Đã cập nhật 'manual_label.csv' với dữ liệu mới được gán nhãn.")

    raw_df = raw_df.drop(query_samples.index).reset_index(drop=True)
    raw_df.to_csv("data/cleaned/vid2_v2.csv", index=False, encoding="utf-8-sig")
    print("📦 Đã cập nhật, còn lại:", len(raw_df), "dòng.")

print("🎉 Hoàn tất tất cả các vòng Active Learning.")



🔁 Vòng Active Learning 1
📤 Đã xuất 50 comment chưa gán nhãn vào data/active_learning/query_round_1.csv
✅ Đã cập nhật 'manual_label.csv' với dữ liệu mới được gán nhãn.
📦 Đã cập nhật, còn lại: 6262 dòng.

🔁 Vòng Active Learning 2
📤 Đã xuất 50 comment chưa gán nhãn vào data/active_learning/query_round_2.csv
✅ Đã cập nhật 'manual_label.csv' với dữ liệu mới được gán nhãn.
📦 Đã cập nhật, còn lại: 6212 dòng.

🔁 Vòng Active Learning 3
📤 Đã xuất 50 comment chưa gán nhãn vào data/active_learning/query_round_3.csv
✅ Đã cập nhật 'manual_label.csv' với dữ liệu mới được gán nhãn.
📦 Đã cập nhật, còn lại: 6162 dòng.

🔁 Vòng Active Learning 4
📤 Đã xuất 50 comment chưa gán nhãn vào data/active_learning/query_round_4.csv
✅ Đã cập nhật 'manual_label.csv' với dữ liệu mới được gán nhãn.
📦 Đã cập nhật, còn lại: 6112 dòng.

🔁 Vòng Active Learning 5
📤 Đã xuất 50 comment chưa gán nhãn vào data/active_learning/query_round_5.csv
✅ Đã cập nhật 'manual_label.csv' với dữ liệu mới được gán nhãn.
📦 Đã cập nhật, còn l

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import joblib
from datetime import datetime


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


df = pd.read_csv("data/manual/vid3_v1 (1).csv").dropna(subset=["Comment", "Label"])
X_text = df["Comment"].astype(str)
y = df["Label"].astype(int)


vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(X_text)


model = LogisticRegression(max_iter=1000)
model.fit(X, y)


y_pred = model.predict(X)
print("📊 Kết quả trên toàn bộ tập đã gán nhãn:")
print(classification_report(y, y_pred))

joblib.dump(model, f"model/final_model_{timestamp}.pkl")
joblib.dump(vectorizer, f"model/vectorizer_{timestamp}.pkl")


📊 Kết quả trên toàn bộ tập đã gán nhãn:
              precision    recall  f1-score   support

           0       0.78      0.93      0.85       334
           1       0.89      0.80      0.84       215
           2       0.86      0.65      0.74       178

    accuracy                           0.82       727
   macro avg       0.84      0.79      0.81       727
weighted avg       0.83      0.82      0.82       727



['model/vectorizer_20250519_164634.pkl']

In [ ]:
import pandas as pd
import joblib


model = joblib.load("model/final_model_20250519_164634.pkl")
vectorizer = joblib.load("model/vectorizer_20250519_164634.pkl")

unlabeled_df = pd.read_csv("data/all.csv").dropna(subset=["Comment"])

df_before = unlabeled_df.iloc[:5159].copy()
df_after = unlabeled_df.iloc[5159:].copy()

X_after_text = df_after["Comment"].astype(str)
X_after_vectorized = vectorizer.transform(X_after_text)


pred_labels = model.predict(X_after_vectorized)

df_after["Label"] = pred_labels

final_df = pd.concat([df_before, df_after], ignore_index=True)

final_df.to_csv("data/result/file_v2.csv", index=False, encoding="utf-8-sig")
